# FakeRecogna (2022) com XGBoost

Use uma cópia deste repositório. O exemplo executa TF-IDF na divisão 42. Para repetir as seis representações nas cinco divisões, habilite a execução completa e escolha GPU no Colab. Os três modelos contextuais baixam pesos públicos. Se a sessão for temporária, copie as saídas para armazenamento persistente ao concluir cada etapa.

In [ ]:
from pathlib import Path
import sys
import subprocess

CODIGO = Path.cwd()
if not (CODIGO / 'requirements.txt').is_file():
    CODIGO = Path('/content/tg-desinformacao')
    if not CODIGO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/diego-henrique-vaz/avaliacao-de-representacoes-vetoriais-na-identificacao-de-desinformacao', str(CODIGO)], check=True)
sys.path.insert(0, str(CODIGO))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODIGO / 'requirements.txt')], check=True)


In [ ]:
from reconstituir_fakerecogna import reconstituir
from apoio_fakerecogna_20261110 import carregar, dividir

PASTA = Path('execucao_fakerecogna').resolve()
EXECUTAR_ESTUDO_COMPLETO = False
reconstituir(PASTA)
dados = carregar(PASTA)
treino, teste = dividir(PASTA, dados, 42)
print(f'{len(dados)} textos, {dados.group_id.nunique()} grupos')


In [ ]:
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from apoio_fakerecogna_20261110 import avaliar, tokenizar

inicio = time.perf_counter()
tfidf = TfidfVectorizer(tokenizer=tokenizar, token_pattern=None, lowercase=False,
                       max_features=5000, ngram_range=(1, 2), max_df=.85, min_df=2,
                       dtype=np.float32)
treino_tfidf = tfidf.fit_transform(dados.iloc[treino].text)
teste_tfidf = tfidf.transform(dados.iloc[teste].text)
avaliar(PASTA, dados, treino, teste, 'tfidf', treino_tfidf, teste_tfidf,
        {'seed': 42, 'fitted_on_train_only': True, 'vocabulary_size': len(tfidf.vocabulary_),
         'representation_seconds': time.perf_counter() - inicio});


In [ ]:
from apoio_fakerecogna_20261110 import tradicionais, contextual, consolidar

if EXECUTAR_ESTUDO_COMPLETO:
    tradicionais(PASTA, dados)
    for metodo in ('bertimbau', 'e5', 'qwen3'):
        contextual(PASTA, dados, metodo)
    consolidar(PASTA, final=True)
